# Modeling Experiments.

This notebook summarizes the results generated by `src/models/run_experiments.py`. We will start from the saved artifacts: metrics, summary, predictions, and models to:

1. Visualize and compare metrics by model/fold.
2. Analyze the errors on the test set (residuals, and y_true vs y_pred scatter).
3. Review feature importance (if available for the model) and document conclusions.

### 1. Setup
Update the `EXPERIMENT_DIR` path with the specific folder (e.g., `experiment_YYYYMMDD_HHMMSS`).

In [37]:
import json
from pathlib import Path

import pandas as pd

import plotly.express as px
import plotly.graph_objects as go

EXPERIMENT_BASE_DIR = Path("../data/results/modeling/experiments")
runner_experiments = sorted(
    [d for d in EXPERIMENT_BASE_DIR.iterdir() if d.is_dir() and d.name.startswith("runner_id_")],
    reverse=True,
)
if not runner_experiments:
    raise FileNotFoundError("No runner_id experiments found. Run run_experiments.py first.")
EXPERIMENT_DIR = runner_experiments[0]
print(f"Using experiment directory: {EXPERIMENT_DIR.name}")
metrics_path = EXPERIMENT_DIR / "metrics.csv"
summary_path = EXPERIMENT_DIR / "summary.csv"
predictions_path = EXPERIMENT_DIR / "predictions.parquet"
config_path = EXPERIMENT_DIR / "config.json"
feature_cols_path = EXPERIMENT_DIR / "feature_columns.json"
models_dir = EXPERIMENT_DIR  

metrics_df = pd.read_csv(metrics_path)
summary_df = pd.read_csv(summary_path)
pred_df = pd.read_parquet(predictions_path)
if feature_cols_path.exists():
    feature_columns = json.loads(feature_cols_path.read_text())
else:
    feature_columns = None

if config_path.exists():
    config = json.loads(config_path.read_text())
    target_name = config.get("target", "reported_rpe")
else:
    config = None
    target_name = "reported_rpe"

target_pretty_map = {
    "reported_rpe": "RPE",
    "Fatigue_Score": "Fatigue Score",
    "fatigue_level": "Fatigue Level",
}
target_label = target_pretty_map.get(target_name, target_name)

summary_df

Using experiment directory: runner_id_20251113_114446


,model,split,mae_mean,mae_std,rmse_mean,rmse_std,r2_mean,r2_std,med_ae_mean,med_ae_std,max_err_mean,max_err_std,samples_total
0,catboost,cv,0.054843,0.008650,0.073212,0.014698,0.666401,0.121547,0.041604,0.005225,0.307540,0.046844,12595
1,catboost,test,0.049522,NaN,0.063753,NaN,0.785023,NaN,0.040855,NaN,0.288940,NaN,2954
2,elasticnet,cv,0.061869,0.005093,0.087415,0.014094,0.527705,0.174452,0.047160,0.005604,1.039618,0.916838,12595
3,elasticnet,test,0.060726,NaN,0.080793,NaN,0.654752,NaN,0.044843,NaN,0.381028,NaN,2954
4,gradient_boosting,cv,0.054804,0.010496,0.073347,0.017010,0.664793,0.133878,0.041578,0.004645,0.322899,0.068663,12595
5,gradient_boosting,test,0.049743,NaN,0.064123,NaN,0.782523,NaN,0.040651,NaN,0.290121,NaN,2954
6,hist_gradient_boosting,cv,0.053812,0.008019,0.073135,0.013756,0.668383,0.116116,0.039157,0.004237,0.308598,0.043780,12595
7,hist_gradient_boosting,test,0.049964,NaN,0.066424,NaN,0.766634,NaN,0.038775,NaN,0.293525,NaN,2954
8,random_forest,cv,0.055299,0.008664,0.074506,0.013252,0.654242,0.127603,0.040952,0.005621,0.317142,0.040400,12595
9,random_forest,test,0.059239,NaN,0.081675,NaN,0.647166,NaN,0.041262,NaN,0.379530,NaN,2954


### 2. Metrics Comparison
Charts to compare MAE/RMSE/R² by model and split.

In [38]:
fig = px.bar(
    summary_df,
    x="model",
    y="mae_mean",
    color="split",
    error_y="mae_std",
    title="MAE medio por modelo y split",
)
fig.show()

fig = px.bar(
    summary_df,
    x="model",
    y="rmse_mean",
    color="split",
    error_y="rmse_std",
    title="RMSE medio por modelo y split",
)
fig.show()

fig = px.bar(
    summary_df,
    x="model",
    y="r2_mean",
    color="split",
    error_y="r2_std",
    title="R² medio por modelo y split",
)
fig.show()

### 3. Residuals and scatters (test)

We inspect how each model performs on the test set.

In [39]:
pred_df["residual"] = pred_df["y_true"] - pred_df["y_pred"]

scatter_title = f"Dispersión {target_label} real vs predicho (test)"
fig = px.scatter(
    pred_df,
    x="y_true",
    y="y_pred",
    color="model",
    title=scatter_title,
    labels={"y_true": f"{target_label} real", "y_pred": f"{target_label} predicho"},
)
fig.add_trace(
    go.Scatter(
        x=[pred_df.y_true.min(), pred_df.y_true.max()],
        y=[pred_df.y_true.min(), pred_df.y_true.max()],
        mode="lines",
        name="Ideal",
    )
)
fig.show()

fig = px.box(
    pred_df,
    x="model",
    y="residual",
    title=f"Distribución de residuos por modelo (test) - {target_label}",
)
fig.show()

### 4. Feature Importance




In [40]:
import joblib
import numpy as np

available_models = sorted(summary_df["model"].unique())
for model_name in available_models:
    model_path = models_dir / f"{model_name}_best.joblib"
    if not model_path.exists():
        print(f"Model artifact not found for {model_name}.")
        continue

    pipeline = joblib.load(model_path)
    model = pipeline.named_steps["model"]

    importances = getattr(model, "feature_importances_", None)
    if importances is None:
        coef = getattr(model, "coef_", None)
        if coef is not None:
            importances = np.abs(np.ravel(coef))
        else:
            print(f"{model_name} does not expose feature importances or coefficients.")
            continue

    feature_names = (
        feature_columns
        if feature_columns is not None and len(feature_columns) == len(importances)
        else [f"f{i}" for i in range(len(importances))]
    )

    fi = (
        pd.DataFrame({"feature": feature_names, "importance": importances})
        .sort_values("importance", ascending=False)
        .head(20)
    )

    px.bar(fi, x="feature", y="importance", title=f"Top features ({model_name})").show()


hist_gradient_boosting does not expose feature importances or coefficients.
